import week 3 module files

In [33]:
import sys
import os

# Path to Week 3 src folder
week3_path = os.path.abspath("../../week_03_herschel_bulkley/src")

sys.path.append(week3_path)

Install Dependancies

In [34]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath("../src"))

from full_system_hydraulics import (
    drillpipe_pressure_loss,
    bit_pressure_loss,
    annular_pressure_loss,
    standpipe_pressure
)

# import previous models
# from herschel_bulkley_model import *
# from hb_hydraulics import *

from hb_hydraulics import (
    pressure_gradient_hb,
    annular_shear_rate
)
from herschel_bulkley_model import (
    apparent_viscosity_hb,
    herschel_bulkley_shear_stress,
    temperature_adjusted_K,
    pressure_gradient_hb
)

Define System Parameters

In [35]:
depth = 2500
# --- Flow Parameters ---
flow_rate = 0.03
pipe_diameter = 0.127
hole_diameter = 0.216
pipe_length = depth
nozzle_area = 0.0008
# --- Rheology Parameters (DEFINE THESE FIRST) ---

# From your corrected HB estimation
K_surface = 0.025     # Consistency index (Pa·s^n)
n = 0.58              # Flow behavior index
yield_stress = 7.18   # Pa
mud_density = 1200   # kg/m3

# Temperature parameters
surface_temp = 30
temp = 120        # example downhole temp
Ea = 5000         # activation energy (adjust as needed)


Compute Velocities

In [36]:
pipe_area = np.pi * pipe_diameter**2 / 4
pipe_velocity = flow_rate / pipe_area

annular_area = np.pi * (hole_diameter**2 - pipe_diameter**2) / 4
annular_velocity = flow_rate / annular_area

Use HB Rheology

In [37]:
shear_rate = annular_shear_rate(
    flow_rate,
    hole_diameter,
    pipe_diameter
)

K_downhole = temperature_adjusted_K(
    K_surface,
    Ea,
    surface_temp,
    temp
)

mu = apparent_viscosity_hb(
    yield_stress,
    K_downhole,
    n,
    shear_rate
)

Compute Pressure Losses

Drillpipe

In [38]:
dp_pipe = drillpipe_pressure_loss(
    mu,
    pipe_velocity,
    pipe_diameter,
    pipe_length
)

print("Drill Pipe Pressure Loss:",dp_pipe, 'Pa')

Drill Pipe Pressure Loss: 775521.3817536948 Pa


Bit

In [39]:
dp_bit = bit_pressure_loss(
    flow_rate,
    mud_density,
    nozzle_area
)

print("Bit Pressure Loss:", dp_bit, 'Pa')

Bit Pressure Loss: 843750.0 Pa


Annulus

In [40]:
dp_dz = pressure_gradient_hb(
    mu,
    annular_velocity,
    hole_diameter - pipe_diameter
)

dp_annulus = annular_pressure_loss(dp_dz, depth)

print("Annular Pressure Loss:", dp_annulus, 'Pa')

Annular Pressure Loss: 834342.7951577854 Pa


Compute SPP

In [41]:
spp = standpipe_pressure(
    dp_pipe,
    dp_bit,
    dp_annulus
)

print("Standpipe Pressure:", spp, 'Pa') # Pascals (Pa)

Standpipe Pressure: 2453614.1769114803 Pa
